# Modèle logit multiple - Base P-COLD

Mémoire M2 EGRF - Détection et pilotage du risque opérationnel bancaire

## Objectif

On cherche à modéliser la probabilité qu'un événement de perte opérationnelle soit **une fraude** (interne ou externe) plutôt qu'un autre type d'événement bâlois. La variable dépendante est donc **binaire** :

$$Y_i = \begin{cases} 1 & \text{si l'événement } i \text{ est une fraude (interne ou externe)} \\ 0 & \text{sinon} \end{cases}$$

Ce notebook suit pas à pas les étapes de la méthodologie : construction de la variable cible, construction des variables explicatives, estimation par maximum de vraisemblance, puis lecture des résultats (coefficients, odds ratios, qualité d'ajustement).

In [ ]:
# Librairies nécessaires
# - pandas / numpy : manipulation des données
# - statsmodels : pour avoir une sortie de régression complète façon "économétrie"
#   (contrairement à scikit-learn, statsmodels donne directement les p-values,
#   écarts-types, intervalles de confiance, etc., ce qui est plus adapté ici)
# - sklearn.metrics : pour la courbe ROC et la matrice de confusion
# - matplotlib : graphiques

import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

plt.rcParams['figure.figsize'] = (8, 5)

In [ ]:
# On repart de la base déjà nettoyée à l'étape d'apurement (pcold_clean.csv)
df = pd.read_csv('pcold_clean.csv')
print(f"{len(df)} événements chargés.")
df.head()

## 1. Construction de la variable dépendante

On regroupe les 5 catégories de la variable `Event_type_grouped` en 2 classes :
- **1 (fraude)** : Internal fraud + External fraud
- **0 (non-fraude)** : toutes les autres catégories

`.isin([...])` renvoie `True`/`False` pour chaque ligne, et `.astype(int)` transforme ça en 1/0, directement exploitable par le modèle.

In [ ]:
df['Y_fraude'] = df['Event_type_grouped'].isin(['Internal fraud', 'External fraud']).astype(int)

print("Répartition de la variable cible :")
print(df['Y_fraude'].value_counts(normalize=True).round(3))
# On voit ici la proportion d'événements de fraude (1) vs autres (0) dans la base.
# C'est utile pour juger si la classe est déséquilibrée.

## 2. Construction des variables explicatives


Une première tentative avec la variable `Causal factor` (facteur causal : People, Process, System, External event) donnait des résultats *trop* bons (pseudo R² = 0,54). En creusant, on découvre pourquoi : dans la taxonomie bâloise, le facteur causal **détermine presque mécaniquement** le type d'événement.

**On retire donc `Causal factor` des variables explicatives**, et on garde :
- `Business line` (ligne métier, variable catégorielle)
- `log_loss` (montant de la perte, transformé en log)

In [ ]:
# Vérification de la fuite de données : tableau croisé Causal factor x Event type
pd.crosstab(df['Causal factor'], df['Event_type_grouped'])
# -> on voit clairement que 'External event' est presque toujours associé à 'External fraud',
#    et 'People' très majoritairement à 'Internal fraud'. C'est cette évidence qui a motivé
#    le retrait de Causal factor du modèle.

In [ ]:
variables_explicatives = ['Business line', 'log_loss']

# On garde uniquement les lignes complètes sur ces variables (pas de valeurs manquantes)
df_modele = df[variables_explicatives + ['Y_fraude']].dropna()
print(f"N utilisées pour le modèle logit : {len(df_modele)}")

# pd.get_dummies transforme la variable catégorielle 'Business line' en plusieurs
# colonnes binaires (une par modalité). drop_first=True retire une modalité de
# référence (nécessaire pour éviter la colinéarité parfaite entre les dummies).
X = pd.get_dummies(df_modele[['Business line']], drop_first=True).astype(float)
X['log_loss'] = df_modele['log_loss']

# sm.add_constant ajoute la colonne de 1 correspondant à la constante beta_0 du modèle
X = sm.add_constant(X)

y = df_modele['Y_fraude']

X.head()

## 3. Estimation par maximum de vraisemblance

`sm.Logit(y, X).fit()` estime le vecteur de paramètres β qui maximise la log-vraisemblance :

$$\log L(\beta) = \sum_i \left[ y_i \log F(X_i\beta) + (1-y_i)\log(1 - F(X_i\beta)) \right]$$

`method='newton'` indique d'utiliser l'algorithme de Newton-Raphson pour la résolution itérative, comme décrit dans la méthodologie.

In [ ]:
modele_logit = sm.Logit(y, X)
resultats = modele_logit.fit(method='newton', maxiter=100)

# .summary() donne une sortie complète : coefficients, écarts-types, z-stats,
# p-values, intervalles de confiance, pseudo R², log-vraisemblance, etc.
print(resultats.summary())

### Comment lire ce tableau

- **coef** : les β estimés. Seul le signe est directement interprétable (positif = augmente la probabilité de fraude, négatif = la diminue).
- **P>|z|** : la p-value du test de significativité individuelle (test de Wald). En dessous de 0,05, le coefficient est jugé significatif.
- **Pseudo R-squ.** : équivalent du R² du modèle linéaire, mais construit à partir des log-vraisemblances (voir méthodologie).
- **LLR p-value** : p-value du test du rapport de vraisemblance, comparant le modèle complet au modèle réduit à la seule constante.

## 4. Odds ratios

Les coefficients bruts sont difficiles à interpréter directement. On les transforme en **odds ratios** :

$$OR = \exp(\beta_k)$$

Un OR > 1 signifie que la variable augmente les chances (odds) de fraude ; un OR < 1 signifie qu'elle les diminue. Par exemple, un OR de 2 signifie que les chances de fraude sont multipliées par 2 lorsque la variable augmente d'une unité (ou change de modalité pour une variable catégorielle).

In [ ]:
# np.exp() sur les coefficients donne directement les odds ratios
odds_ratios = np.exp(resultats.params)

# .conf_int() donne l'intervalle de confiance à 95% des coefficients ; on l'exponentie
# aussi pour obtenir l'intervalle de confiance des odds ratios
ic = np.exp(resultats.conf_int())

tableau_or = pd.DataFrame({
    'Odds Ratio': odds_ratios,
    'IC 2.5%': ic[0],
    'IC 97.5%': ic[1],
    'p-value': resultats.pvalues
})
tableau_or.round(4)

la variable `log_loss` a un OR d'environ 1,45 et est très significative un montant de perte plus élevé est associé à une probabilité de fraude plus forte. La modalité `Retail banking` a un OR très supérieur à 1, ce qui indique que les événements en banque de détail sont nettement plus souvent des fraudes que dans la catégorie de référence.

## 5. Qualité d'ajustement et pouvoir prédictif

- **Pseudo R² de McFadden** : mesure globale d'ajustement (voir méthodologie, section diagnostics).
- **Taux de prédictions correctes** : proportion d'événements bien classés, en utilisant la règle $\hat y_i = 1$ si $\hat p_i > 0{,}5$.

In [ ]:
print(f"Pseudo R2 (McFadden) : {resultats.prsquared:.4f}")

# .predict(X) donne les probabilités prédites p_hat_i ; on applique le seuil à 0.5
probabilites_predites = resultats.predict(X)
y_pred = (probabilites_predites > 0.5).astype(int)

taux_correct = (y_pred == y).mean()
print(f"Taux de prédictions correctes : {taux_correct:.4f}")

## 6. Matrice de confusion et courbe ROC

La matrice de confusion détaille les quatre cas possibles (vrai positif, faux positif, vrai négatif, faux négatif). La courbe ROC, elle, évalue la capacité du modèle à discriminer les deux classes sur l'ensemble des seuils possibles (pas seulement 0,5), ce qui est particulièrement utile en présence de déséquilibre entre classes.

In [ ]:
# Matrice de confusion
matrice = confusion_matrix(y, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=matrice, display_labels=['Non-fraude', 'Fraude'])
disp.plot(cmap='Greys', values_format='d')
plt.title("Matrice de confusion - Modèle logit")
plt.show()

In [ ]:
# Courbe ROC : on fait varier le seuil de décision et on trace
# le taux de vrais positifs contre le taux de faux positifs
taux_fp, taux_vp, seuils = roc_curve(y, probabilites_predites)
auc = roc_auc_score(y, probabilites_predites)

plt.plot(taux_fp, taux_vp, color='#2c2c2c', label=f'Courbe ROC (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], color='#a0a0a0', linestyle='--', label='Modèle aléatoire')
plt.xlabel("Taux de faux positifs")
plt.ylabel("Taux de vrais positifs")
plt.title("Courbe ROC - Modèle logit")
plt.legend()
plt.tight_layout()
plt.show()

print(f"AUC : {auc:.4f}")
# Un AUC proche de 1 indique un excellent pouvoir discriminant,
# un AUC proche de 0.5 indique un modèle proche du hasard.

## 7. Test du rapport de vraisemblance

Ce test compare la log-vraisemblance du modèle complet à celle du modèle nul (constante seule), afin de vérifier que l'ensemble des variables explicatives apporte globalement une information significative.

$$-2\ln(LR) = 2(\ln L_c - \ln L_{nc}) \sim \chi^2(q)$$

In [ ]:
print(f"Log-vraisemblance (modèle complet) : {resultats.llf:.2f}")
print(f"Log-vraisemblance (modèle nul)     : {resultats.llnull:.2f}")
print(f"Statistique LR                     : {resultats.llr:.2f}")
print(f"p-value du test LR                 : {resultats.llr_pvalue:.6f}")

if resultats.llr_pvalue < 0.05:
    print(">> Le modèle complet apporte une amélioration significative par rapport au modèle nul.")